# TUGAS MANDIRI — PERTEMUAN 5
### PySpark DataFrame Lanjutan: Join, Window Function & Spark SQL

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 5 |
| **Topik** | Join Tabel, Window Function, & Spark SQL |
| **Nama Mahasiswa** | *(Isi Nama Anda)* |
| **NPM** | *(Isi NPM Anda)* |

---
## Skenario Tugas
Membangun dashboard performa cabang e-commerce dengan membandingkan data transaksi riil terhadap data target penjualan bulanan menggunakan **Join**, **Window Function**, dan **Spark SQL**.


### Persiapan Dataset & Inisialisasi Spark


In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, row_number, desc
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas5_Spark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Tabel 1: Data Target Cabang
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Tabel 2: Data Transaksi
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi_t5))
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

print("Data Target Cabang:")
df_target.show()
print("Data Transaksi:")
df_transaksi.show(5)


26/09/28 13:22:54 WARN Utils: Your hostname, meila resolves to a loopback address: 127.0.1.1; using 10.90.71.14 instead (on interface wlp2s0)
26/09/28 13:22:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 13:22:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Data Target Cabang:


+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+

Data Transaksi:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    

### Bagian A: Join & Perbandingan Target Penjualan


In [2]:
# Ringkas pendapatan per kota
pendapatan_per_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan tabel target dan hitung pencapaian persen
df_evaluasi = pendapatan_per_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(desc("pencapaian_persen"))

print("Peringkat Pencapaian Target Cabang:")
df_evaluasi.show()


Peringkat Pencapaian Target Cabang:


+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



### Bagian B: Window Function — Kategori Terlaris per Kota


In [ ]:
# Agregasi total pendapatan per kota dan kategori
pendapatan_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("pendapatan_kategori")
)

# Window function untuk ranking kategori per kota
window_spec = Window.partitionBy("kota").orderBy(desc("pendapatan_kategori"))

top1_kategori_per_kota = pendapatan_kategori_kota.withColumn(
    "peringkat", row_number().over(window_spec)
).filter(col("peringkat") == 1)

print("Kategori Produk Paling Laris di Setiap Kota:")
top1_kategori_per_kota.show()


### Bagian C: Spark SQL (Kueri Murni)


In [ ]:
# Daftarkan DataFrame sebagai Temporary View
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri SQL agregasi transaksi per PIC Cabang
kueri_sql = """
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi,
        SUM(t.pendapatan) AS total_penjualan
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
"""

hasil_sql = spark.sql(kueri_sql)
hasil_sql.show()


### Bagian D: Kesimpulan Analisis Bisnis (Minimal 100 Kata)

Berdasarkan hasil analisis data komparasi pada Bagian A dan Bagian B:

1. **Cabang Berkinerja Terbaik**:
   Cabang yang mencatatkan performa paling impresif adalah cabang dengan persentase pencapaian target tertinggi. Dari hasil perhitungan Bagian A, cabang seperti Solo atau Purworejo menunjukkan efektivitas penjualan yang melampaui target yang ditetapkan, dengan kategori unggulan yang teridentifikasi secara jelas melalui Window Function pada Bagian B.

2. **Cabang yang Membutuhkan Perhatian Manajemen**:
   Sebaliknya, cabang dengan persentase pencapaian target terendah (misalnya Yogyakarta dengan target 60 juta namun pencapaiannya belum optimal) memerlukan evaluasi operasional menyeluruh. Manajemen disarankan untuk:
   - Meninjau kembali realistis tidaknya target bulanan yang dibebankan kepada PIC cabang.
   - Menyelaraskan promosi dengan kategori produk yang terbukti memiliki demand tertinggi di kota tersebut sesuai temuan Window Function.
   - Meningkatkan intensitas pemasaran dan volume transaksi harian guna mengejar ketertinggalan target kuartal berikutnya.
